# Synthetic reviews generated with a small language model

Generates one review for each (item, star rating) pair of the dataset with Qwen2.5-0.5B served by
[Ollama](https://ollama.com) (`ollama pull qwen2.5:0.5b`), and saves them to
`preprocessing/<dataset>_llm.parquet`, used by the `slm` ablation in `ReLoG_ablations.ipynb`.
Items without metadata, and failed generations, fall back to the static template.

In [ ]:
import os
import threading
from concurrent.futures import ThreadPoolExecutor, as_completed

import pandas as pd
from tqdm import tqdm

In [ ]:
DATASET = 'cd'   # movies | cd | electronics
NAME = {'movies': 'Movies_and_TV', 'cd': 'CD_Vinyl', 'electronics': 'electronics'}[DATASET]


def find_parquet(fname):
    for root in ['preprocessing', '../preprocessing']:
        p = os.path.join(root, fname)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(fname)


df_sampled = pd.read_parquet(find_parquet(f'{NAME}_review.parquet'))
df_meta_aligned = pd.read_parquet(find_parquet(f'{NAME}_meta.parquet'))

In [ ]:
MODEL_NAME     = "qwen2.5:0.5b"
CACHE_PATH     = os.path.join(os.path.dirname(find_parquet(f'{NAME}_review.parquet')), f'{NAME}_llm.parquet')
GEN_TEMPERATURE = 0.3
GEN_SEED        = 42
MAX_NEW_TOKENS  = 130
CHECKPOINT_EVERY = 20

N_WORKERS = 8

DEBUG_LLM_ERRORS = False

def get_prompt_descriptor(meta_text, max_words=50):
    if not isinstance(meta_text, str) or not meta_text.strip():
        return ""
    return " ".join(meta_text.split()[:max_words])

prompt_desc_map = dict(
    zip(df_meta_aligned['item_id_int'],
        df_meta_aligned['meta_text'].map(get_prompt_descriptor))
)

def get_title(meta_text, max_words=50):
    if not isinstance(meta_text, str) or not meta_text.strip():
        return ""
    title = meta_text.split('.')[0].strip()
    return " ".join(title.split()[:max_words])

title_map = dict(
    zip(df_meta_aligned['item_id_int'],
        df_meta_aligned['meta_text'].map(get_title))
)

SENTIMENT_TEMPLATES = {
    5: ["I absolutely love this. It's excellent and exactly what I hoped for. Highly recommended.",
        "Fantastic product, I'm extremely satisfied. It works perfectly and I would buy it again."],
    4: ["A good product that I'm happy with. It works well, with only minor drawbacks.",
        "Solid and reliable, I like it. Not perfect, but it does what it promises."],
    3: ["This product is okay. It's average: it does the job but nothing special.",
        "Mixed feelings about this one. It's decent but it has a few shortcomings."],
    2: ["I'm disappointed with this product. It has problems and didn't meet my expectations.",
        "Not great. It works poorly and I expected more from it."],
    1: ["This product is terrible. It doesn't work as expected and I would not recommend it.",
        "Very poor quality, I regret buying it. It failed to do what it should."],
}

def template_fallback(stars, item_id):
    variants = SENTIMENT_TEMPLATES[stars]
    sentiment = variants[int(item_id) % len(variants)]
    return f"{sentiment} {title_map.get(int(item_id), '')}".strip()

SYSTEM_PROMPT = (
    "You are a customer writing a product review for an e-commerce website. "
    "Given a product and the star rating you gave it, write a short, realistic "
    "review of 2-3 sentences that is clearly consistent with that rating "
    "(enthusiastic for 5 stars, negative for 1 star, mixed for 3). "
    "Write your OWN opinion in your OWN words: do NOT copy or repeat the product "
    "description, specifications, or technical details verbatim. "
    "STRICT LIMIT: no more than 30 words total, never exceed it. "
    "Output ONLY the review text: no preamble, no quotes, no rating number, no lists, "
    "no reasoning, no explanation of your choice."
)

def build_prompt(stars, descriptor):
    return (f"Product: {descriptor}\n"
            f"My rating: {stars} out of 5 stars.\n\n"
            f"Write my review:")

_CONTROL_TOKENS = [
    "<|im_end|>", "<|im_start|>", "<end_of_turn>", "<start_of_turn>",
    "<|end_of_turn|>", "<|start_of_turn|>", "<|think|>", "<|/think|>",
    "<|eot_id|>", "<|start_header_id|>", "<|end_header_id|>", "<|begin_of_text|>",
    "<br>", "<br/>", "<br />",
]

def clean_review(text):
    if not text:
        return ""
    t = text
    for tok in _CONTROL_TOKENS:
        t = t.replace(tok, "")
    lower = t.lower()
    marker = "done thinking."
    if marker in lower:
        idx = lower.index(marker) + len(marker)
        t = t[idx:]
    t = " ".join(t.strip().split())
    t = t.strip().strip('"').strip("'").strip()
    for p in ("review:", "here is a review:", "here's a review:",
              "sure,", "certainly,", "okay,"):
        if t.lower().startswith(p):
            t = t[len(p):].strip()
    words = t.split()
    if len(words) > 100:
        t = " ".join(words[:100])
    return t

FEWSHOT_EXAMPLES = [
    {"role": "user", "content": "Product: Wireless Bluetooth Earbuds, sweat resistant.\nMy rating: 5 out of 5 stars.\n\nWrite my review:"},
    {"role": "assistant", "content": "These earbuds completely exceeded my expectations! Comfortable fit, great sound, and they never fall out during workouts."},
    {"role": "user", "content": "Product: USB Charging Cable, braided nylon.\nMy rating: 2 out of 5 stars.\n\nWrite my review:"},
    {"role": "assistant", "content": "Disappointed with this cable. It stopped charging properly after two weeks of light use. Wouldn't recommend it."},
]


_PLACEHOLDER_MARKERS = ("description unavailable", "no description available",
                        "product description not available", "n/a")

_REFUSAL_MARKERS = ("i'm sorry", "i cannot", "i can't assist",
                    "as an ai", "i am unable", "i don't have enough")

import ollama

def _extract_content(resp):
    try:
        return resp["message"]["content"]
    except (TypeError, KeyError):
        return resp.message.content

def _looks_copied(review_text, descriptor, min_shared_words=8):
    review_words = review_text.lower().split()
    descriptor_words = descriptor.lower().split()
    max_run = 0
    desc_set_windows = set()
    for i in range(len(descriptor_words) - min_shared_words + 1):
        desc_set_windows.add(tuple(descriptor_words[i:i+min_shared_words]))
    for i in range(len(review_words) - min_shared_words + 1):
        if tuple(review_words[i:i+min_shared_words]) in desc_set_windows:
            return True
    return False

def generate_review_llm(stars, item_id):
    descriptor = prompt_desc_map.get(int(item_id), "")

    if not descriptor.strip() or any(m in descriptor.lower() for m in _PLACEHOLDER_MARKERS):
        return template_fallback(stars, item_id), "fallback"

    try:
        resp = ollama.chat(
            model=MODEL_NAME,
            messages=[{"role": "system", "content": SYSTEM_PROMPT}]
                     + FEWSHOT_EXAMPLES
                     + [{"role": "user", "content": build_prompt(stars, descriptor)}],
            think=False,
            options={"temperature": GEN_TEMPERATURE,
                     "seed": GEN_SEED,
                     "num_predict": MAX_NEW_TOKENS},
        )

        done_reason = resp.get("done_reason")
        if done_reason == "length":
            raise ValueError(f"generation truncated from server (done_reason=length)")

        text = clean_review(_extract_content(resp))

        if len(text.split()) < 3:
            raise ValueError(f"too short output: {text!r}")

        if any(m in text.lower() for m in _REFUSAL_MARKERS):
            raise ValueError(f"meta/refusal response detected: {text!r}")

        if _looks_copied(text, descriptor):
            raise ValueError(f"copy-paste output of product, no review: {text!r}")

        return text, "llm"
    except Exception as e:
        if DEBUG_LLM_ERRORS:
            import traceback
            print(f"\n[DEBUG] Fallback triggered for item={item_id}, stars={stars}")
            print(f"[DEBUG] Exception: {type(e).__name__}: {e}")
            traceback.print_exc()
        return template_fallback(stars, item_id), "fallback"

try:
    _ = ollama.chat(model=MODEL_NAME,
                    messages=[{"role": "user", "content": "ping"}],
                    options={"num_predict": 1})
    print(f"Ollama OK — model '{MODEL_NAME}' reachable.")
except Exception as e:
    raise RuntimeError(
        f"Unable to contact Ollama with the model '{MODEL_NAME}'.\n"
        f"Make sure the server is running and that the model has been downloaded "
        f"('ollama pull {MODEL_NAME}'). Error: {e}"
    )

In [ ]:
def _stars(r):
    return max(1, min(5, int(round(float(r)))))

if os.path.exists(CACHE_PATH):
    _cache_df = pd.read_parquet(CACHE_PATH)
    review_cache = {
        (int(r.item_id_int), int(r.rating_int)): (r.review, r.source)
        for r in _cache_df.itertuples(index=False)
    }
    print(f"Cache found: {len(review_cache)} pairs (item, star) already generated.")
else:
    review_cache = {}

pairs = df_sampled[['item_id_int', 'rating']].copy()
pairs['rating_int'] = pairs['rating'].map(_stars)
unique_pairs = pairs[['item_id_int', 'rating_int']].drop_duplicates()
print(f"Unique pairs (item, star) in dataset: {len(unique_pairs)}")

def _save_cache():
    df_out = pd.DataFrame(
        [{"item_id_int": k[0], "rating_int": k[1], "review": v[0], "source": v[1]}
         for k, v in review_cache.items()]
    )
    df_out.to_parquet(CACHE_PATH, index=False)


todo = [(int(i), int(s)) for i, s in unique_pairs.itertuples(index=False)
        if (int(i), int(s)) not in review_cache]
print(f"To be generated: {len(todo)} / {len(unique_pairs)} pairs "
      f"(already in cache: {len(unique_pairs) - len(todo)})")

new_count = 0
_save_lock = threading.Lock()

if todo:
    try:
        with ThreadPoolExecutor(max_workers=N_WORKERS) as executor:
            future_to_key = {
                executor.submit(generate_review_llm, stars, item_id): (item_id, stars)
                for item_id, stars in todo
            }
            for future in tqdm(as_completed(future_to_key), total=len(todo),
                               desc=f"LLM reviews [{MODEL_NAME}] x{N_WORKERS} workers"):
                item_id, stars = future_to_key[future]
                review_cache[(item_id, stars)] = future.result()
                new_count += 1
                if new_count % CHECKPOINT_EVERY == 0:
                    with _save_lock:
                        _save_cache()
    finally:
        if new_count > 0:
            with _save_lock:
                _save_cache()
            print(f"[CHECKPOINT] Saved {new_count} new reviews before exit.")

if new_count > 0:
    print(f"Generated {new_count} new reviews. Cache updated: {CACHE_PATH}")
else:
    print("No new reviews to generate (all cached).")